<a href="https://colab.research.google.com/github/ek7anna/AI-TASKS/blob/main/graph_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

In [3]:
dataset = Planetoid(root='data/Cora', name='Cora')

data = dataset[0]

print(dataset)
print(data)

Processing...


Cora()
Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


Done!


In [4]:
print("Dataset:", dataset)

print("Number of graphs:", len(dataset))
print("Number of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)

print("Number of node features:", dataset.num_features)
print("Number of classes:", dataset.num_classes)

print("Node feature matrix shape:", data.x.shape)
print("Edge index shape:", data.edge_index.shape)
print("Labels shape:", data.y.shape)

Dataset: Cora()
Number of graphs: 1
Number of nodes: 2708
Number of edges: 10556
Number of node features: 1433
Number of classes: 7
Node feature matrix shape: torch.Size([2708, 1433])
Edge index shape: torch.Size([2, 10556])
Labels shape: torch.Size([2708])


In [5]:
class GCN(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(dataset.num_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):

        x = data.x
        edge_index = data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(x, training=self.training)

        x = self.conv2(x, edge_index)

        return x

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GCN().to(device)

data = data.to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

print(model)

GCN(
  (conv1): GCNConv(1433, 16)
  (conv2): GCNConv(16, 7)
)


In [7]:
def train():

    model.train()

    optimizer.zero_grad()

    out = model(data)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()

    optimizer.step()

    return loss.item()

In [8]:
def test():

    model.eval()

    out = model(data)

    pred = out.argmax(dim=1)

    correct = (
        pred[data.test_mask]
        ==
        data.y[data.test_mask]
    ).sum()

    acc = int(correct) / int(data.test_mask.sum())

    return acc

In [9]:
for epoch in range(1, 201):

    loss = train()

    if epoch % 20 == 0:

        acc = test()

        print(
            f"Epoch {epoch:03d} | Loss {loss:.4f} | Accuracy {acc:.4f}"
        )

Epoch 020 | Loss 0.3197 | Accuracy 0.8020
Epoch 040 | Loss 0.0673 | Accuracy 0.7950
Epoch 060 | Loss 0.0577 | Accuracy 0.7960
Epoch 080 | Loss 0.0618 | Accuracy 0.8110
Epoch 100 | Loss 0.0328 | Accuracy 0.8080
Epoch 120 | Loss 0.0418 | Accuracy 0.8060
Epoch 140 | Loss 0.0299 | Accuracy 0.8120
Epoch 160 | Loss 0.0193 | Accuracy 0.8130
Epoch 180 | Loss 0.0278 | Accuracy 0.8070
Epoch 200 | Loss 0.0250 | Accuracy 0.8150


In [10]:
model.eval()

out = model(data)

pred = out.argmax(dim=1)

print("Predicted class:", pred[0].item())
print("Actual class:", data.y[0].item())

Predicted class: 3
Actual class: 3
